# Stopping-rule ablation: when to stop a boosting fit

Boosting adds one base-learner update per step, and the number of steps is the
stopping iteration. Too few steps underfit; too many overfit. This ablation compares
eight stopping rules on the same data. Each rule chooses the stopping iteration, and
the chosen model is scored on held-out rows that the rule never saw.

The question is which rule gives the lowest held-out loss, and whether the choice
holds on real data as well as on synthetic data. The second version of the study
(`stopping-ablation/2`) replaces the October 12 run. The first run's stopping
conclusions are withdrawn, and its audit is kept in the same directory.

The full report is `docs/benchmarks/stopping_ablation_509/v2/README.md`. The
per-seed comparisons and group summaries are in `summary.json`, and this page
recomputes its tables from that file.

## Stopping rules compared

The rules share one set of development rows and one first budget. They differ in how
they choose the stopping iteration and in how the preprocessing is built.

| Rule | How it chooses the stopping iteration | Preprocessing |
|---|---|---|
| `fixed_argmin` | the iteration that minimises the loss on a single split; the historical reference | refitted |
| `fixed_preprocessing` | the same as `fixed_argmin`, on the matched native route | native, fitted once |
| `boundary_growth` | starts from the first budget and grows it when the minimum sits at the boundary | native |
| `patience_60` | stops when the loss has not improved for 60 steps | native |
| `repeated_cv_growth` | grows the budget with repeated cross-validation | native |
| `fold_local_fixed` | a fixed budget on each fold's training rows | fitted on each fold's training rows |
| `fold_local_growth` | grows the budget within each fold | fitted on each fold's training rows |
| `fold_local_cv_growth` | grows the budget with cross-validation within each fold | fitted on each fold's training rows |

Two routes matter for deployment. The native rules fit the preprocessing once on all
development rows, so the held-out rows can influence the spline boundaries and the
factor levels. The fold-local rules fit the preprocessing again on each fold's training
rows. The report calls the native comparisons diagnostic, and it draws its
deployment conclusions from the fold-local rules.

Each comparison is a pair: a rule and its reference. A negative change favours the
rule. Every change is a percentage of the reference's loss on the same panel, method,
tier and seed.

In [1]:
import json
from pathlib import Path

import polars as pl
from great_tables import GT

# This page runs from docs/book/benchmarks; the summary sits in the v2 directory.
SUMMARY = Path("evidence/stopping_rule_ablation/summary.json")
summary = json.loads(SUMMARY.read_text())
print(summary["experiment_version"], "-", summary["interpretation"])
print("comparison groups:", list(summary["comparisons"].keys()))

stopping-ablation/2 - descriptive paired assessment seeds; no confidence intervals
comparison groups: ['preprocessing', 'native_growth', 'native_patience', 'native_validation', 'fold_local_growth', 'fold_local_validation']


## Each comparison across all cells

A cell is one panel, method and tier, with the median change taken over its seeds.
The table summarises each comparison group over its 28 cells. It shows how often the
rule lost to its reference and by how much at worst.

In [2]:
group_labels = {
    "preprocessing": "Preprocessing route (native, diagnostic)",
    "native_growth": "Budget growth (native)",
    "native_patience": "Patience (native)",
    "native_validation": "Repeated-CV growth (native)",
    "fold_local_growth": "Budget growth (fold-local)",
    "fold_local_validation": "Repeated-CV growth (fold-local)",
}

group_rows = []
for key, label in group_labels.items():
    cells = pl.DataFrame(summary["comparisons"][key]["summary"])
    # A cell with no finite seed carries no comparison; none are present in this run.
    finite = cells.filter(pl.col("n_finite") > 0)
    group_rows.append(
        {
            "Comparison": label,
            "Rule": finite["rule"][0],
            "Reference": finite["reference"][0],
            "Cells": finite.height,
            "Median cell change": finite["median_relative"].median(),
            "Cells where rule is worse": int((finite["median_relative"] > 0).sum()),
            "Cells where rule is better": int((finite["median_relative"] < 0).sum()),
            "Largest single-seed increase": finite["max_relative"].max(),
        }
    )

group_table = pl.DataFrame(group_rows)
(
    GT(group_table)
    .tab_header(
        title="Stopping rules against their references, all panels",
        subtitle="Change in held-out loss, percent of the reference; negative favours the rule",
    )
    .fmt_percent(
        columns=["Median cell change", "Largest single-seed increase"],
        scale_values=True,
        decimals=1,
    )
    .tab_stubhead(label="Comparison")
    .tab_source_note(
        "Cells: panel, method and tier, each the median over its seeds. Largest single-seed increase: the worst "
        "seed's percent increase in any cell. Descriptive; no confidence intervals. Source: summary.json."
    )
)

GT(_tbl_data=shape: (6, 8)
┌────────────┬────────────┬────────────┬───────┬────────────┬────────────┬────────────┬────────────┐
│ Comparison ┆ Rule       ┆ Reference  ┆ Cells ┆ Median     ┆ Cells      ┆ Cells      ┆ Largest    │
│ ---        ┆ ---        ┆ ---        ┆ ---   ┆ cell       ┆ where rule ┆ where rule ┆ single-see │
│ str        ┆ str        ┆ str        ┆ i64   ┆ change     ┆ is worse   ┆ is better  ┆ d increase │
│            ┆            ┆            ┆       ┆ ---        ┆ ---        ┆ ---        ┆ ---        │
│            ┆            ┆            ┆       ┆ f64        ┆ i64        ┆ i64        ┆ f64        │
╞════════════╪════════════╪════════════╪═══════╪════════════╪════════════╪════════════╪════════════╡
│ Preprocess ┆ fixed_prep ┆ fixed_argm ┆ 28    ┆ 0.0        ┆ 6          ┆ 2          ┆ 0.020154   │
│ ing route  ┆ rocessing  ┆ in         ┆       ┆            ┆            ┆            ┆            │
│ (native,   ┆            ┆            ┆       ┆            ┆            ┆            ┆            │
│ d…         ┆            ┆            ┆       ┆            ┆            ┆            ┆            │
│ Budget     ┆ boundary_g ┆ fixed_prep ┆ 28    ┆ 0.0        ┆ 0          ┆ 12         ┆ 0.081616   │
│ growth     ┆ rowth      ┆ rocessing  ┆       ┆            ┆            ┆            ┆            │
│ (native)   ┆            ┆            ┆       ┆            ┆            ┆            ┆            │
│ Patience   ┆ patience_6 ┆ boundary_g ┆ 28    ┆ 0.0        ┆ 4          ┆ 0          ┆ 0.204936   │
│ (native)   ┆ 0          ┆ rowth      ┆       ┆            ┆            ┆            ┆            │
│ Repeated-C ┆ repeated_c ┆ boundary_g ┆ 28    ┆ -0.000142  ┆ 9          ┆ 15         ┆ 1.290956   │
│ V growth   ┆ v_growth   ┆ rowth      ┆       ┆            ┆            ┆            ┆            │
│ (native)   ┆            ┆            ┆       ┆            ┆            ┆            ┆            │
│ Budget     ┆ fold_local ┆ fold_local ┆ 28    ┆ 0.0        ┆ 0          ┆ 12         ┆ 0.113498   │
│ growth (fo ┆ _growth    ┆ _fixed     ┆       ┆            ┆            ┆            ┆            │
│ ld-local)  ┆            ┆            ┆       ┆            ┆            ┆            ┆            │
│ Repeated-C ┆ fold_local ┆ fold_local ┆ 28    ┆ -0.000227  ┆ 9          ┆ 16         ┆ 1.074482   │
│ V growth   ┆ _cv_growth ┆ _growth    ┆       ┆            ┆            ┆            ┆            │
│ (fold-loca ┆            ┆            ┆       ┆            ┆            ┆            ┆            │
│ l…         ┆            ┆            ┆       ┆            ┆            ┆            ┆            │
└────────────┴────────────┴────────────┴───────┴────────────┴────────────┴────────────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x10f3c9610>, _boxhead=Boxhead([ColInfo(var='Comparison', type=<ColInfoTypeEnum.default: 1>, column_label='Comparison', column_align='left', column_width=None), ColInfo(var='Rule', type=<ColInfoTypeEnum.default: 1>, column_label='Rule', column_align='left', column_width=None), ColInfo(var='Reference', type=<ColInfoTypeEnum.default: 1>, column_label='Reference', column_align='left', column_width=None), ColInfo(var='Cells', type=<ColInfoTypeEnum.default: 1>, column_label='Cells', column_align='right', column_width=None), ColInfo(var='Median cell change', type=<ColInfoTypeEnum.default: 1>, column_label='Median cell change', column_align='right', column_width=None), ColInfo(var='Cells where rule is worse', type=<ColInfoTypeEnum.default: 1>, column_label='Cells where rule is worse', column_align='right', column_width=None), ColInfo(var='Cells where rule is better', type=<ColInfoTypeEnum.default: 1>, column_label='Cells where rule is better', column_align='right', column_width=None), ColInfo(var='Largest single-seed increase', type=<ColInfoTypeEnum.default: 1>, column_label='Largest single-seed increase', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object

The fold-local budget growth rule never raised the median loss across seeds in any of the 28 cells. Its
largest single-seed increase is about 11%. The fold-local repeated-CV growth rule
lowers the loss in 16 cells and raises it in 9. Its largest increase is 107%, on one
seed. The native patience rule is worse than its reference in 4 cells, with a largest
increase of 20%. The native repeated-CV growth rule is worse in 9 cells, with an
increase of 129% on one seed.

## Deployment-facing cells

The fold-local comparisons are the ones the report uses for deployment decisions. The
next table puts the two fold-local rules side by side for each cell. Each entry is the
median change against that rule's reference: fold-local fixed budget for growth, and
fold-local growth for repeated-CV growth. Bold marks the lower value.

In [3]:
def cell_key(row):
    return (row["task"], row["method"], row["tier"])

growth = {cell_key(r): r for r in summary["comparisons"]["fold_local_growth"]["summary"]}
cv = {cell_key(r): r for r in summary["comparisons"]["fold_local_validation"]["summary"]}

def bold_lower(a, b_value):
    """Format two percent changes with the lower one in bold."""
    fa, fb = f"{a * 100:+.1f}%", f"{b_value * 100:+.1f}%"
    if a < b_value:
        return f"**{fa}**", fb
    if b_value < a:
        return fa, f"**{fb}**"
    return f"**{fa}**", f"**{fb}**"

cell_rows = []
for key in sorted(growth):
    g_text, cv_text = bold_lower(growth[key]["median_relative"], cv[key]["median_relative"])
    cell_rows.append(
        {
            "Panel": key[0],
            "Method": key[1],
            "Tier": key[2],
            "Budget growth": g_text,
            "Repeated-CV growth": cv_text,
        }
    )

cell_table = pl.DataFrame(cell_rows)
(
    GT(cell_table)
    .tab_header(
        title="Fold-local stopping rules, median change in held-out loss per cell",
        subtitle="Percent change; negative favours the rule; budget growth vs fold-local fixed, repeated-CV growth vs fold-local growth",
    )
    .fmt_markdown(columns=["Budget growth", "Repeated-CV growth"])
    .tab_stubhead(label="Panel")
    .tab_source_note(
        "Bold: the lower change in the row. Each change is the median over the cell's seeds. Source: summary.json, "
        "comparisons fold_local_growth and fold_local_validation."
    )
)

GT(_tbl_data=shape: (28, 5)
┌────────────────┬─────────────────┬───────┬───────────────┬────────────────────┐
│ Panel          ┆ Method          ┆ Tier  ┆ Budget growth ┆ Repeated-CV growth │
│ ---            ┆ ---             ┆ ---   ┆ ---           ┆ ---                │
│ str            ┆ str             ┆ str   ┆ str           ┆ str                │
╞════════════════╪═════════════════╪═══════╪═══════════════╪════════════════════╡
│ bike_future    ┆ mboost_additive ┆ small ┆ +0.0%         ┆ **-4.1%**          │
│ bike_future    ┆ mboost_additive ┆ tuned ┆ +0.0%         ┆ **-2.3%**          │
│ bike_future    ┆ mboost_hybrid   ┆ small ┆ **+0.0%**     ┆ +1.0%              │
│ bike_future    ┆ mboost_hybrid   ┆ tuned ┆ **+0.0%**     ┆ +0.9%              │
│ bike_iid       ┆ mboost_additive ┆ small ┆ **-2.9%**     ┆ +0.0%              │
│ …              ┆ …               ┆ …     ┆ …             ┆ …                  │
│ joint_boundary ┆ mboost_hybrid   ┆ tuned ┆ -1.2%         ┆ **-1.4%**          │
│ smooth_sparse  ┆ mboost_additive ┆ small ┆ +0.0%         ┆ **-0.3%**          │
│ smooth_sparse  ┆ mboost_additive ┆ tuned ┆ +0.0%         ┆ **-0.1%**          │
│ smooth_sparse  ┆ mboost_hybrid   ┆ small ┆ **+0.0%**     ┆ +0.1%              │
│ smooth_sparse  ┆ mboost_hybrid   ┆ tuned ┆ **+0.0%**     ┆ +0.1%              │
└────────────────┴─────────────────┴───────┴───────────────┴────────────────────┘, _body=<great_tables._gt_data.Body object at 0x10e0ead80>, _boxhead=Boxhead([ColInfo(var='Panel', type=<ColInfoTypeEnum.default: 1>, column_label='Panel', column_align='left', column_width=None), ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Tier', type=<ColInfoTypeEnum.default: 1>, column_label='Tier', column_align='left', column_width=None), ColInfo(var='Budget growth', type=<ColInfoTypeEnum.default: 1>, column_label='Budget growth', column_align='left', column_width=None), ColInfo(var='Repeated-CV growth', type=<ColInfoTypeEnum.default: 1>, column_label='Repeated-CV growth', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10e2ef320>, _spanners=Spanners([]), _heading=Heading(title='Fold-local stopping rules, median change in held-out loss per cell', subtitle='Percent change; negative favours the rule; budget growth vs fold-local fixed, repeated-CV growth vs fold-local growth', preheader=None), _stubhead='Panel', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10e2ed9d0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10e2ef500>, _source_notes=["Bold: the lower change in the row. Each change is the median over the cell's seeds. Source: summary.json, comparisons fold_local_growth and fold_local_validation."], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10f5a7f20>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10f5a7e60>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True

On the fold-local route, budget growth lowers the loss in 12 of 28 cells and leaves it
unchanged in the other 16. The repeated-CV growth rule has the mixed record, with its
largest losses on the housing and cancer panels. The worst single-seed cases are
listed next.

## Largest single-seed increases

The table lists the five largest single-seed increases among the fold-local rules, with
the seed where each occurred. These are the cases a deployment should check first.

In [4]:
worst = []
for group in ["fold_local_growth", "fold_local_validation"]:
    for r in summary["comparisons"][group]["summary"]:
        if r["n_finite"] == 0:
            continue
        worst.append(
            {
                "Rule": r["rule"],
                "Panel": r["task"],
                "Method": r["method"],
                "Tier": r["tier"],
                "Seed": r["worst_seed"],
                "Reference loss": r["worst_reference_loss"],
                "Rule loss": r["worst_test_loss"],
                "Change": (r["worst_test_loss"] - r["worst_reference_loss"]) / r["worst_reference_loss"],
            }
        )

worst_table = pl.DataFrame(worst).sort("Change", descending=True).head(5)
(
    GT(worst_table)
    .tab_header(
        title="Five largest single-seed increases under fold-local stopping",
        subtitle="Held-out loss; the rule's loss against the reference's loss at the worst seed",
    )
    .fmt_number(columns=["Reference loss", "Rule loss"], decimals=4)
    .fmt_percent(columns=["Change"], scale_values=True, decimals=1)
    .tab_stubhead(label="Rule")
    .tab_source_note("Worst seed per cell as recorded in summary.json. Descriptive; one seed per row.")
)

GT(_tbl_data=shape: (5, 8)
┌─────────────────┬─────────────┬────────────────┬───────┬──────┬───────────┬───────────┬──────────┐
│ Rule            ┆ Panel       ┆ Method         ┆ Tier  ┆ Seed ┆ Reference ┆ Rule loss ┆ Change   │
│ ---             ┆ ---         ┆ ---            ┆ ---   ┆ ---  ┆ loss      ┆ ---       ┆ ---      │
│ str             ┆ str         ┆ str            ┆ str   ┆ i64  ┆ ---       ┆ f64       ┆ f64      │
│                 ┆             ┆                ┆       ┆      ┆ f64       ┆           ┆          │
╞═════════════════╪═════════════╪════════════════╪═══════╪══════╪═══════════╪═══════════╪══════════╡
│ fold_local_cv_g ┆ housing_iid ┆ mboost_additiv ┆ tuned ┆ 2    ┆ 0.078806  ┆ 0.163482  ┆ 1.074482 │
│ rowth           ┆             ┆ e              ┆       ┆      ┆           ┆           ┆          │
│ fold_local_cv_g ┆ housing_iid ┆ mboost_additiv ┆ small ┆ 2    ┆ 0.082537  ┆ 0.145742  ┆ 0.765788 │
│ rowth           ┆             ┆ e              ┆       ┆      ┆           ┆           ┆          │
│ fold_local_cv_g ┆ cancer      ┆ mboost_hybrid  ┆ tuned ┆ 0    ┆ 0.09756   ┆ 0.14085   ┆ 0.443727 │
│ rowth           ┆             ┆                ┆       ┆      ┆           ┆           ┆          │
│ fold_local_grow ┆ cancer      ┆ mboost_hybrid  ┆ tuned ┆ 2    ┆ 0.350084  ┆ 0.389818  ┆ 0.113498 │
│ th              ┆             ┆                ┆       ┆      ┆           ┆           ┆          │
│ fold_local_cv_g ┆ cancer      ┆ mboost_additiv ┆ tuned ┆ 1    ┆ 0.064207  ┆ 0.067308  ┆ 0.048302 │
│ rowth           ┆             ┆ e              ┆       ┆      ┆           ┆           ┆          │
└─────────────────┴─────────────┴────────────────┴───────┴──────┴───────────┴───────────┴──────────┘, _body=<great_tables._gt_data.Body object at 0x10f5a7470>, _boxhead=Boxhead([ColInfo(var='Rule', type=<ColInfoTypeEnum.default: 1>, column_label='Rule', column_align='left', column_width=None), ColInfo(var='Panel', type=<ColInfoTypeEnum.default: 1>, column_label='Panel', column_align='left', column_width=None), ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Tier', type=<ColInfoTypeEnum.default: 1>, column_label='Tier', column_align='left', column_width=None), ColInfo(var='Seed', type=<ColInfoTypeEnum.default: 1>, column_label='Seed', column_align='right', column_width=None), ColInfo(var='Reference loss', type=<ColInfoTypeEnum.default: 1>, column_label='Reference loss', column_align='right', column_width=None), ColInfo(var='Rule loss', type=<ColInfoTypeEnum.default: 1>, column_label='Rule loss', column_align='right', column_width=None), ColInfo(var='Change', type=<ColInfoTypeEnum.default: 1>, column_label='Change', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10e2eda60>, _spanners=Spanners([]), _heading=Heading(title='Five largest single-seed increases under fold-local stopping', subtitle="Held-out loss; the rule's loss against the reference's loss at the worst seed", preheader=None), _stubhead='Rule', _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10e2edeb0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10f5d8ad0>, _source_notes=['Worst seed per cell as recorded in summary.json. Descriptive; one seed per row.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10f630890>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10f5dae70>, <great_tables._gt_data.FormatInfo object at 0x10f630a70>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, cate

## Findings

- **Growing the budget within each fold is the one rule with no losses in these
  cells.** The fold-local growth rule has no higher median loss than its fixed-budget reference
  in 28 cells, and it is better in 12. Its worst single seed rises by about 11%.
- **Repeated-CV growth on fold-local data helps in most cells and hurts in some.** It
  lowers the loss in 16 of 28 cells. In 9 cells it raises the loss, and one seed of one
  cell rises by 107%.
- **Patience and repeated-CV growth on native preprocessing are diagnostic.** Their
  held-out rows enter the basis construction, so their numbers are not
  deployment-facing. The native patience rule is worse in 4 cells.
- **The earlier stopping claims do not hold.** The October 12 run's conclusions are
  withdrawn, and the correction report lists the seeds that contradicted them.

## Limits

- **No public default changes.** The experiment changes nothing in the library. Its
  output is a set of measurements, not a recommendation to replace a default.
- **No confidence intervals.** The report gives medians, ranges and worst seeds.
  Real-data assessment splits overlap, and temporal origins share training history, so
  the real-data panels are not independent experiments.
- **Patience has no fold-local form.** The public API refuses fold-local patience.
  Its result is a native diagnostic only.
- **Tree topology is out of scope.** The report does not test tree-form or adaptive
  degrees-of-freedom rules.

## Reproduction

The report's reproduction section gives the commands. The run used the release
extension built from commit `ebe1e45`, and the full results are in
`v2/results.json.gz`, a 20 MB file that this page does not read.